# Figure 4A, 4B & Figure S8: Within- and Cross-Region Feedback-to-Cue HFA Representational Similarity Analysis (RSA)

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig 4A (within-ROI ACC-ACC and HPC-HPC old vs new RSA, all ps > 0.05), Fig 4B (between-ROI ACC feedback -> HPC cue RSA, old vs new cluster p = 0.003), Fig S8 (dlPFC -> HPC, dmPFC -> HPC, OFC -> HPC feedback-to-cue RSA, all ps > 0.05) |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/projects/guLab/Salman/EphysAnalyses/{subj}/scratch/TFR/{feedback_start,cue_on}-tfr.h5`, `Day1_reref_elec_df`, `learn_df_RWH_v2.csv`, `full_df_RWH_v2.csv` |
| **Outputs** | `$MEMORYBANDIT_DATA_DIR/scratch/MemoryBandit/RSA_MVTemplate/{trial_level,group_level}/*`, `results/rsa_controls/z_cluster/*.pdf` |
| **Dependencies** | `membandit`, `mne`, `numpy`, `pandas`, `scipy`, `matplotlib` |
| **Expected runtime** | ~30-60 minutes for batch 250-shuffle subject RSA + 5,000-permutation group cluster tests across 6 ROI pairs |
| **Upstream steps** | `03_spectral/00_compute_TFR_Fig2A_5A.ipynb` |


In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to $MEMORYBANDIT_DATA_DIR)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import glob
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.spatial.distance import pdist, squareform
from scipy.stats import wilcoxon, spearmanr, pearsonr
from matplotlib.backends.backend_pdf import PdfPages
from mne.stats import permutation_cluster_1samp_test
from membandit.plotting_style import set_plot_style
from membandit.io import Subject
from membandit._impl.multivariate import load_subject_crossregion_hfa, _get_smoothed_event_signal, build_crossregion_rsa_paths, build_rpe_category_model_rdm, compute_cross_temporal_rsa_from_hfa, compute_cross_temporal_rsa_partial_from_hfa, compute_cross_temporal_reinstatement_category_maps, run_subject_crossregion_rsa_permutation, run_group_crossregion_rsa_pipeline, build_cluster_mask_from_clusters, load_z_diff_stack_from_pair_dir, run_group_crossregion_rsa_cluster_test, compute_cross_temporal_reinstatement_maps, _padded_row_zscore_cosine_similarity, _compute_pairwise_dissimilarity, _canonicalize_rpe_cat_label

warnings.filterwarnings('ignore')
set_plot_style()


# Cross-Region RSA (MVTemplate-Aligned)

This notebook keeps RSA-specific computation, while aligning shared data handling with `Subject`, `_get_smoothed_event_signal`, and legacy `combined_df` trial mapping semantics from `HFA_RPE_RSA.ipynb`.


In [ ]:
# =============================
# Paths + subject discovery
# =============================
from membandit import paths as mpaths
base_dir = mpaths.get_base_dir()
load_dir = mpaths.get_emu_behavior_dir()

demo_file = mpaths.get_subjects_xlsx_path()
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)]
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

subjects = subj_df.MSSMCode.astype(str).tolist()
print(f'Found {len(subjects)} analyzable subjects')
print(subjects)


In [ ]:
print(subjects)

In [ ]:
list(zip(np.arange(len(subjects))+1, subjects))

In [ ]:
# =============================
# Analysis configuration
# =============================
ROI_PAIRS = [('ACC', 'ACC'), ('HPC', 'HPC'),
             ('ACC', 'HPC'), ('dmPFC', 'HPC'), ('dlPFC', 'HPC'),
             ('OFC', 'HPC')]
DEFAULT_CUE_ROI = 'HPC'
ALLOW_NON_HPC_CUE = True

EVENT_ENCODING = 'feedback_start'
EVENT_CUE = 'cue_on'

FREQ_RANGE = (70., 201.)
WIN_MS = 200
DOWNSAMPLE_FACTOR = 5
MIN_TRIALS = 10

RDM_METHOD = 'euclidean'
RDM_COMPARE_METHOD = 'rho-a'
RSA_SHUFFLE_TYPE = 'trial_labels'
RSA_N_PERM = 250
RSA_SEED = 42

RUN_SINGLE_SUBJECT_SMOKE = True
RUN_EXAMPLE_PAIR_PLOT = True
RUN_BATCH_RSA = True
RUN_GROUP_CLUSTER = True

EXAMPLE_SELECTION_SCOPE = 'smoke_pair'
EXAMPLE_SUBJECT = None
EXAMPLE_FB_TIME = None
EXAMPLE_CUE_TIME = None
EXAMPLE_TOP_K_PAIRS = 10

N_JOBS_OUTER = min(8, os.cpu_count() or 1)
OVERWRITE_TRIAL_LEVEL = True
VERBOSE = True

RSA_TRIAL_SAVE_DIR = f'{base_dir}/scratch/MemoryBandit/RSA_MVTemplate/trial_level'
RSA_GROUP_SAVE_DIR = f'{base_dir}/scratch/MemoryBandit/RSA_MVTemplate/group_level'

SMOKE_SUBJECT = 'MS036' if 'MS036' in subjects else subjects[0]
SMOKE_PAIR = ROI_PAIRS[0]
GROUP_PLOT_PAIR = ROI_PAIRS[0]

if (not ALLOW_NON_HPC_CUE):
    for _, cue_roi in ROI_PAIRS:
        assert cue_roi == DEFAULT_CUE_ROI, f'Cue ROI {cue_roi} is not allowed when ALLOW_NON_HPC_CUE=False'

print('Smoke subject:', SMOKE_SUBJECT)
print('Smoke pair:', SMOKE_PAIR)
print('Example selection scope:', EXAMPLE_SELECTION_SCOPE)


## Single-Subject Smoke

In [ ]:
# =============================
# Example plotting helpers
# =============================
def compute_example_rdm_at_timepoint(data, timepoint_idx, epoch='encoding', memory_condition=None, electrode_idx=None, rdm_method=None):
    """RDM at one timepoint; uses same metric as the RSA pipeline (see RDM_METHOD)."""
    if rdm_method is None:
        rdm_method = RDM_METHOD
    if epoch == 'encoding':
        hfa = np.asarray(data['encoding_hfa'], dtype=float)
    elif epoch == 'cue':
        hfa = np.asarray(data['cue_hfa'], dtype=float)
    else:
        raise ValueError(f'Unsupported epoch: {epoch}')

    patterns = hfa[:, :, timepoint_idx]
    if electrode_idx is not None:
        patterns = patterns[:, [int(electrode_idx)]]

    if memory_condition is not None:
        mask = np.asarray(data['memory_outcomes']) == memory_condition
        patterns = patterns[mask]

    if patterns.shape[0] < 2:
        return np.full((patterns.shape[0], patterns.shape[0]), np.nan, dtype=float)

    method = str(rdm_method).lower()
    if method == 'corr':
        method = 'correlation'
    if patterns.shape[1] == 1 and method == 'correlation':
        print('WARNING: single-channel RDM; correlation distance undefined — using euclidean')
        method = 'euclidean'
    if method in ('mahalanobis',):
        cov = np.cov(patterns, rowvar=False)
        cov = np.atleast_2d(cov)
        vi = np.linalg.pinv(cov)
        return np.asarray(squareform(pdist(patterns, metric='mahalanobis', VI=vi)), dtype=float)
    return np.asarray(squareform(pdist(patterns, metric=method)), dtype=float)


def plot_example_rdm(rdm, ax=None, cmap='viridis', show_colorbar=False,
                     show_labels=False, title=None, vmin=None, vmax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    else:
        fig = ax.figure

    im = ax.imshow(rdm, cmap=cmap, aspect='equal', vmin=vmin, vmax=vmax)
    if not show_labels:
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.set_xlabel('Trials')
        ax.set_ylabel('Trials')

    if title:
        ax.set_title(title, fontsize=10)

    if show_colorbar:
        plt.colorbar(im, ax=ax, shrink=0.8, label='Dissimilarity')

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    return fig, ax


def compute_single_channel_pair_rsa(data, encoding_electrode_idx, cue_electrode_idx,
                                    rdm_method='euclidean', rdm_compare_method='rho-a'):
    return compute_cross_temporal_rsa_from_hfa(
        encoding_hfa=np.asarray(data['encoding_hfa'])[:, [int(encoding_electrode_idx)], :],
        cue_hfa=np.asarray(data['cue_hfa'])[:, [int(cue_electrode_idx)], :],
        memory_outcomes=np.asarray(data['memory_outcomes']),
        rdm_method=rdm_method,
        rdm_compare_method=rdm_compare_method,
    )


def rank_example_subjects_for_pair(trial_save_dir, group_save_dir, roi_pair):
    pair_paths = build_crossregion_rsa_paths(
        trial_save_dir=trial_save_dir,
        group_save_dir=group_save_dir,
        encoding_roi=roi_pair[0],
        cue_roi=roi_pair[1],
        subject=None,
    )
    rows = []
    for npz_path in sorted(glob.glob(os.path.join(pair_paths['pair_dir'], '*_rsa.npz'))):
        with np.load(npz_path, allow_pickle=True) as npz:
            remembered_heatmap = np.asarray(npz['remembered_heatmap'], dtype=float)
            rows.append({
                'subject': str(npz['subject']) if 'subject' in npz.files else os.path.basename(npz_path).split('_')[0],
                'npz_path': npz_path,
                'remembered_peak': float(np.nanmax(remembered_heatmap)),
                'remembered_mean': float(np.nanmean(remembered_heatmap)),
                'n_remembered': int(npz['n_remembered']) if 'n_remembered' in npz.files else np.nan,
                'n_forgotten': int(npz['n_forgotten']) if 'n_forgotten' in npz.files else np.nan,
                'n_trials': int(npz['n_trials']) if 'n_trials' in npz.files else np.nan,
            })
    if len(rows) == 0:
        return pd.DataFrame(columns=['subject', 'npz_path', 'remembered_peak', 'remembered_mean', 'n_remembered', 'n_forgotten', 'n_trials'])
    return pd.DataFrame(rows).sort_values(
        ['remembered_peak', 'remembered_mean', 'n_remembered', 'n_trials'],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)


def rank_example_electrode_pairs(data, roi_pair, rdm_method='euclidean', rdm_compare_method='rho-a'):
    rows = []
    enc_labels = np.asarray(data['electrode_labels_encoding'], dtype=str)
    cue_labels = np.asarray(data['electrode_labels_cue'], dtype=str)
    feedback_times = np.asarray(data['feedback_times'], dtype=float)
    cue_times = np.asarray(data['cue_times'], dtype=float)
    memory_outcomes = np.asarray(data['memory_outcomes'])
    n_remembered = int(np.sum(memory_outcomes == 'remembered'))
    n_forgotten = int(np.sum(memory_outcomes == 'forgotten'))

    for enc_idx, enc_label in enumerate(enc_labels):
        for cue_idx, cue_label in enumerate(cue_labels):
            pair_res = compute_single_channel_pair_rsa(
                data,
                encoding_electrode_idx=enc_idx,
                cue_electrode_idx=cue_idx,
                rdm_method=rdm_method,
                rdm_compare_method=rdm_compare_method,
            )
            remembered_heat = np.asarray(pair_res['remembered_heatmap'], dtype=float)
            forgotten_heat = np.asarray(pair_res['forgotten_heatmap'], dtype=float)
            diff_heat = np.asarray(pair_res['diff_heatmap'], dtype=float)
            if remembered_heat.size == 0 or np.all(np.isnan(remembered_heat)):
                continue

            rem_peak_idx = np.unravel_index(np.nanargmax(remembered_heat), remembered_heat.shape)
            diff_peak_idx = np.unravel_index(np.nanargmax(diff_heat), diff_heat.shape)
            rows.append({
                'encoding_roi': roi_pair[0],
                'cue_roi': roi_pair[1],
                'encoding_electrode_idx': int(enc_idx),
                'cue_electrode_idx': int(cue_idx),
                'encoding_electrode': str(enc_label),
                'cue_electrode': str(cue_label),
                'remembered_peak': float(np.nanmax(remembered_heat)),
                'remembered_mean': float(np.nanmean(remembered_heat)),
                'forgotten_peak': float(np.nanmax(forgotten_heat)) if forgotten_heat.size > 0 else np.nan,
                'diff_peak': float(np.nanmax(diff_heat)) if diff_heat.size > 0 else np.nan,
                'n_remembered': n_remembered,
                'n_forgotten': n_forgotten,
                'peak_feedback_idx': int(rem_peak_idx[0]),
                'peak_cue_idx': int(rem_peak_idx[1]),
                'peak_feedback_time': float(feedback_times[rem_peak_idx[0]]),
                'peak_cue_time': float(cue_times[rem_peak_idx[1]]),
                'diff_peak_feedback_idx': int(diff_peak_idx[0]),
                'diff_peak_cue_idx': int(diff_peak_idx[1]),
                'pair_result': pair_res,
            })

    if len(rows) == 0:
        return pd.DataFrame(columns=[
            'encoding_roi', 'cue_roi', 'encoding_electrode_idx', 'cue_electrode_idx',
            'encoding_electrode', 'cue_electrode', 'remembered_peak', 'remembered_mean',
            'forgotten_peak', 'diff_peak', 'n_remembered', 'n_forgotten',
            'peak_feedback_idx', 'peak_cue_idx', 'peak_feedback_time', 'peak_cue_time',
            'diff_peak_feedback_idx', 'diff_peak_cue_idx', 'pair_result',
        ])

    return pd.DataFrame(rows).sort_values(
        ['remembered_peak', 'remembered_mean', 'n_remembered'],
        ascending=[False, False, False],
    ).reset_index(drop=True)


In [ ]:
# =============================
# Debug: Cross-Region HFA Load + Calibration
# =============================
DEBUG_STATUS = 'FAIL'

try:
    dbg_subject_obj = Subject(SMOKE_SUBJECT, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)

    dbg_loaded = load_subject_crossregion_hfa(
        subject_obj=dbg_subject_obj,
        encoding_roi=SMOKE_PAIR[0],
        cue_roi=SMOKE_PAIR[1],
        event_encoding=EVENT_ENCODING,
        event_cue=EVENT_CUE,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
        min_trials=MIN_TRIALS,
    )

    dbg_enc_all, _, _ = _get_smoothed_event_signal(
        subject_obj=dbg_subject_obj,
        roi=SMOKE_PAIR[0],
        event=EVENT_ENCODING,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
    )
    dbg_cue_all, _, _ = _get_smoothed_event_signal(
        subject_obj=dbg_subject_obj,
        roi=SMOKE_PAIR[1],
        event=EVENT_CUE,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
    )

    dbg_combined = dbg_subject_obj.combined_df[
        (dbg_subject_obj.combined_df['condition'] == 'Day1')
        & (dbg_subject_obj.combined_df['participant'] == SMOKE_SUBJECT)
    ].copy()

    needed_cols = ['trials_gamble', 'trials_mem', 'hits']
    missing_cols = [c for c in needed_cols if c not in dbg_combined.columns]
    assert len(missing_cols) == 0, f'missing trial columns: {missing_cols}'

    dbg_combined = dbg_combined.dropna(subset=['trials_gamble', 'trials_mem']).reset_index(drop=True)
    assert len(dbg_combined) > 0, 'no_day1_trials_after_dropna'

    trial_ix_encoding = dbg_combined['trials_gamble'].to_numpy(dtype=np.int64) - 1
    trial_ix_cue = dbg_combined['trials_mem'].to_numpy(dtype=np.int64) - 1

    in_range = (
        (trial_ix_encoding >= 0) & (trial_ix_encoding < dbg_enc_all.shape[0])
        & (trial_ix_cue >= 0) & (trial_ix_cue < dbg_cue_all.shape[0])
    )

    hits_numeric = pd.to_numeric(dbg_combined['hits'], errors='coerce').to_numpy()
    assert np.all(np.isfinite(hits_numeric)), 'hits contains non-numeric/null values after Day1 subset'
    legacy_mem = np.where(hits_numeric.astype(bool), 'remembered', 'forgotten')

    dbg_alignment_df = pd.DataFrame({
        'row_idx': np.arange(len(dbg_combined), dtype=int),
        'trials_gamble': dbg_combined['trials_gamble'].to_numpy(),
        'trials_mem': dbg_combined['trials_mem'].to_numpy(),
        'enc_ix': trial_ix_encoding,
        'cue_ix': trial_ix_cue,
        'in_range': in_range,
        'hits': hits_numeric,
        'memory_outcome_legacy': legacy_mem,
    })

    memory_outcomes = np.asarray(dbg_loaded['memory_outcomes'])
    trial_index = np.asarray(dbg_loaded['trial_index'])
    retrieval_trial_index = np.asarray(dbg_loaded['retrieval_trial_index'])

    assert dbg_loaded['encoding_hfa'].ndim == 3 and dbg_loaded['cue_hfa'].ndim == 3, 'HFA tensors must be 3D.'
    assert dbg_loaded['encoding_hfa'].shape[0] == dbg_loaded['cue_hfa'].shape[0] == len(memory_outcomes) == len(trial_index), 'Trial dimension mismatch.'
    assert len(retrieval_trial_index) == len(trial_index), 'retrieval_trial_index length mismatch.'
    assert dbg_loaded['encoding_hfa'].shape[2] == len(dbg_loaded['feedback_times']), 'feedback_times length mismatch.'
    assert dbg_loaded['cue_hfa'].shape[2] == len(dbg_loaded['cue_times']), 'cue_times length mismatch.'
    assert np.all(np.isfinite(trial_index)), 'trial_index contains non-finite values.'
    assert np.all((trial_index >= 0) & (trial_index < dbg_enc_all.shape[0])), 'trial_index out of encoding bounds.'
    assert np.all((retrieval_trial_index >= 0) & (retrieval_trial_index < dbg_cue_all.shape[0])), 'retrieval_trial_index out of cue bounds.'

    mem_counts = pd.Series(memory_outcomes).value_counts(dropna=False)
    assert ('remembered' in mem_counts.index) and ('forgotten' in mem_counts.index), 'memory_outcomes missing remembered/forgotten.'
    assert int(np.sum(in_range)) >= MIN_TRIALS, f'Aligned in-range trials below MIN_TRIALS ({np.sum(in_range)} < {MIN_TRIALS}).'

    dropped_trials = int((~in_range).sum())
    total_trials = int(len(in_range))
    dropped_frac = (dropped_trials / total_trials) if total_trials > 0 else np.nan
    enc_dup = int(pd.Series(trial_ix_encoding[in_range]).duplicated().sum()) if np.any(in_range) else 0
    cue_dup = int(pd.Series(trial_ix_cue[in_range]).duplicated().sum()) if np.any(in_range) else 0

    known_labels = {'remembered', 'forgotten'}
    unknown_labels = sorted(set(pd.Series(memory_outcomes).dropna().astype(str).unique()) - known_labels)

    if dropped_frac > 0.20:
        print(f'WARNING: High dropped trial fraction: {dropped_trials}/{total_trials} ({dropped_frac:.1%})')
    if enc_dup > 0:
        print(f'WARNING: Duplicate encoding indices among aligned trials: {enc_dup}')
    if cue_dup > 0:
        print(f'WARNING: Duplicate cue indices among aligned trials: {cue_dup}')
    if len(unknown_labels) > 0:
        print(f'WARNING: Unknown memory labels: {unknown_labels}')

    print('--- Cross-Region HFA Debug Diagnostics ---')
    print('subject:', SMOKE_SUBJECT, '| pair:', SMOKE_PAIR)
    print('alignment_mode:', dbg_loaded.get('alignment_mode', 'legacy_combined_df_trial_map'))
    print('encoding_hfa shape:', dbg_loaded['encoding_hfa'].shape)
    print('cue_hfa shape:', dbg_loaded['cue_hfa'].shape)
    print('raw encoding event shape:', dbg_enc_all.shape)
    print('raw cue event shape:', dbg_cue_all.shape)
    print('aligned_trials:', int(np.sum(in_range)))
    print('dropped_trials:', dropped_trials)
    print('n_remembered:', int(np.sum(memory_outcomes == 'remembered')))
    print('n_forgotten:', int(np.sum(memory_outcomes == 'forgotten')))
    print('memory_label_counts:')
    print(mem_counts)
    print('alignment preview (first 15 rows):')
    print(dbg_alignment_df.head(15))

    DEBUG_STATUS = 'PASS'

except Exception as e:
    print('--- Cross-Region HFA Debug Failure ---')
    print('subject:', SMOKE_SUBJECT, '| pair:', SMOKE_PAIR)
    print('DEBUG_STATUS: FAIL')
    print('error:', repr(e))
    raise

print('DEBUG_STATUS:', DEBUG_STATUS)


In [ ]:
# =============================
# Single-subject smoke run
# =============================
smoke_result = None
if RUN_SINGLE_SUBJECT_SMOKE:
    subject_obj = Subject(SMOKE_SUBJECT, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)
    smoke_result = run_subject_crossregion_rsa_permutation(
        subject_obj=subject_obj,
        encoding_roi=SMOKE_PAIR[0],
        cue_roi=SMOKE_PAIR[1],
        event_encoding=EVENT_ENCODING,
        event_cue=EVENT_CUE,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
        min_trials=MIN_TRIALS,
        n_permutations=RSA_N_PERM,
        shuffle_type=RSA_SHUFFLE_TYPE,
        seed=RSA_SEED,
        rdm_method=RDM_METHOD,
        rdm_compare_method=RDM_COMPARE_METHOD,
    )

    smoke_paths = build_crossregion_rsa_paths(
        trial_save_dir=RSA_TRIAL_SAVE_DIR,
        group_save_dir=RSA_GROUP_SAVE_DIR,
        encoding_roi=SMOKE_PAIR[0],
        cue_roi=SMOKE_PAIR[1],
        subject=SMOKE_SUBJECT,
    )
    os.makedirs(smoke_paths['pair_dir'], exist_ok=True)
    np.savez_compressed(
        smoke_paths['subject_npz'],
        remembered_heatmap=smoke_result['remembered_heatmap'],
        forgotten_heatmap=smoke_result['forgotten_heatmap'],
        diff_heatmap=smoke_result['diff_heatmap'],
        z_diff_heatmap=smoke_result['z_diff_heatmap'],
        p_diff_heatmap=smoke_result['p_diff_heatmap'],
        feedback_times=smoke_result['feedback_times'],
        cue_times=smoke_result['cue_times'],
        memory_outcomes=smoke_result['memory_outcomes'],
        trial_index=smoke_result['trial_index'],
        retrieval_trial_index=smoke_result['retrieval_trial_index'],
        subject=smoke_result['subject'],
        encoding_roi=smoke_result['encoding_roi'],
        cue_roi=smoke_result['cue_roi'],
        n_trials=np.int32(smoke_result['n_trials']),
        n_remembered=np.int32(smoke_result['n_remembered']),
        n_forgotten=np.int32(smoke_result['n_forgotten']),
        rdm_method=np.asarray(smoke_result['rdm_method']),
        rdm_compare_method=np.asarray(smoke_result['rdm_compare_method']),
        shuffle_type=np.asarray(smoke_result['shuffle_type']),
        n_permutations=np.int32(smoke_result['n_permutations']),
        n_dropped_alignment=np.int32(smoke_result.get('n_dropped_alignment', 0)),
        alignment_mode=np.asarray(smoke_result.get('alignment_mode', 'legacy_combined_df_trial_map')),
        status=np.asarray(smoke_result['status']),
    )

    print('Smoke NPZ:', smoke_paths['subject_npz'])
    print('diff shape:', smoke_result['diff_heatmap'].shape)
    print('z range:', float(np.nanmin(smoke_result['z_diff_heatmap'])), float(np.nanmax(smoke_result['z_diff_heatmap'])))


## Example Hit Pair Plot

In [ ]:
RSA_GROUP_SAVE_DIR

In [ ]:
# =============================
# Example hit-RSA pair selection + plotting
# =============================
example_subject_rank_df = pd.DataFrame()
example_pair_rank_df = pd.DataFrame()
example_subject = None
example_data = None
example_pair_result = None
example_selected_pair = None

if not RUN_EXAMPLE_PAIR_PLOT:
    print('Set RUN_EXAMPLE_PAIR_PLOT = True to rank and plot an example hit-RSA pair.')
elif EXAMPLE_SELECTION_SCOPE != 'smoke_pair':
    print(f'Unsupported EXAMPLE_SELECTION_SCOPE: {EXAMPLE_SELECTION_SCOPE}')
else:
    # Example ROI-level RDMs use scipy.pdist with RDM_METHOD (same as compute_cross_temporal_rsa_from_hfa).
    assert RDM_METHOD in ('euclidean', 'correlation', 'corr', 'mahalanobis'), RDM_METHOD

    example_subject_rank_df = rank_example_subjects_for_pair(
        trial_save_dir=RSA_TRIAL_SAVE_DIR,
        group_save_dir=RSA_GROUP_SAVE_DIR,
        roi_pair=SMOKE_PAIR,
    )

    if len(example_subject_rank_df) == 0:
        print(f'No cached subject RSA files found for {SMOKE_PAIR[0]}->{SMOKE_PAIR[1]}.')
    else:
        print(f'Ranked cached subjects for {SMOKE_PAIR[0]}->{SMOKE_PAIR[1]} by remembered/hit RSA:')
        display(example_subject_rank_df.head(EXAMPLE_TOP_K_PAIRS))

        if EXAMPLE_SUBJECT is not None:
            match_df = example_subject_rank_df[example_subject_rank_df['subject'] == EXAMPLE_SUBJECT].reset_index(drop=True)
            if len(match_df) == 0:
                print(f'EXAMPLE_SUBJECT={EXAMPLE_SUBJECT} not found in cached smoke-pair results; using top-ranked subject instead.')
                example_subject = str(example_subject_rank_df.iloc[0]['subject'])
            else:
                example_subject = str(match_df.iloc[0]['subject'])
        else:
            example_subject = str(example_subject_rank_df.iloc[0]['subject'])

        print('Selected example subject:', example_subject)

        example_subject_obj = Subject(example_subject, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)
        example_data = load_subject_crossregion_hfa(
            subject_obj=example_subject_obj,
            encoding_roi=SMOKE_PAIR[0],
            cue_roi=SMOKE_PAIR[1],
            event_encoding=EVENT_ENCODING,
            event_cue=EVENT_CUE,
            freq_range=FREQ_RANGE,
            downsample_factor=DOWNSAMPLE_FACTOR,
            win_ms=WIN_MS,
            min_trials=MIN_TRIALS,
        )

        example_pair_rank_df = rank_example_electrode_pairs(
            example_data,
            roi_pair=SMOKE_PAIR,
            rdm_method=RDM_METHOD,
            rdm_compare_method=RDM_COMPARE_METHOD,
        )

        if len(example_pair_rank_df) == 0:
            print('No valid electrode pairs produced finite remembered/hit RSA heatmaps.')
        else:
            cols_to_show = [
                'encoding_electrode', 'cue_electrode', 'remembered_peak', 'remembered_mean',
                'forgotten_peak', 'diff_peak', 'peak_feedback_time', 'peak_cue_time',
                'n_remembered', 'n_forgotten',
            ]
            print(f'Ranked electrode pairs for {example_subject} {SMOKE_PAIR[0]}->{SMOKE_PAIR[1]}:')
            display(example_pair_rank_df[cols_to_show].head(EXAMPLE_TOP_K_PAIRS))

            example_selected_pair = example_pair_rank_df.iloc[0]
            example_pair_result = example_selected_pair['pair_result']
            feedback_times = np.asarray(example_data['feedback_times'], dtype=float)
            cue_times = np.asarray(example_data['cue_times'], dtype=float)

            if EXAMPLE_FB_TIME is None:
                fb_idx = int(example_selected_pair['peak_feedback_idx'])
            else:
                fb_idx = int(np.argmin(np.abs(feedback_times - float(EXAMPLE_FB_TIME))))

            if EXAMPLE_CUE_TIME is None:
                cue_idx = int(example_selected_pair['peak_cue_idx'])
            else:
                cue_idx = int(np.argmin(np.abs(cue_times - float(EXAMPLE_CUE_TIME))))

            rdm_hit_enc = compute_example_rdm_at_timepoint(
                example_data,
                fb_idx,
                epoch='encoding',
                memory_condition='remembered',
                electrode_idx=None,
                rdm_method=RDM_METHOD,
            )
            rdm_miss_enc = compute_example_rdm_at_timepoint(
                example_data,
                fb_idx,
                epoch='encoding',
                memory_condition='forgotten',
                electrode_idx=None,
                rdm_method=RDM_METHOD,
            )
            rdm_hit_cue = compute_example_rdm_at_timepoint(
                example_data,
                cue_idx,
                epoch='cue',
                memory_condition='remembered',
                electrode_idx=None,
                rdm_method=RDM_METHOD,
            )
            rdm_miss_cue = compute_example_rdm_at_timepoint(
                example_data,
                cue_idx,
                epoch='cue',
                memory_condition='forgotten',
                electrode_idx=None,
                rdm_method=RDM_METHOD,
            )

            rdm_arrays = [rdm_hit_enc, rdm_miss_enc, rdm_hit_cue, rdm_miss_cue]
            finite_vals = [np.nanmax(x) for x in rdm_arrays if x.size > 0 and np.any(np.isfinite(x))]
            rdm_vmax = max(finite_vals) if len(finite_vals) > 0 else 1.0

            print(
                'Selected pair:',
                f"{example_selected_pair['encoding_electrode']} -> {example_selected_pair['cue_electrode']}",
                '| hits peak=', round(float(example_selected_pair['remembered_peak']), 4),
                '| feedback t=', round(float(feedback_times[fb_idx]), 3),
                '| cue t=', round(float(cue_times[cue_idx]), 3),
            )

            fig, axes = plt.subplots(2, 2, figsize=(10, 9))
            plot_example_rdm(rdm_hit_enc, ax=axes[0, 0], vmax=rdm_vmax, show_labels=True)
            plot_example_rdm(rdm_miss_enc, ax=axes[0, 1], vmax=rdm_vmax, show_labels=True)
            plot_example_rdm(rdm_hit_cue, ax=axes[1, 0], vmax=rdm_vmax, show_labels=True)
            plot_example_rdm(rdm_miss_cue, ax=axes[1, 1], vmax=rdm_vmax, show_labels=True)
            im = axes[1, 1].images[0]

            axes[0, 0].set_title(f'Hits (n={rdm_hit_enc.shape[0]})', fontsize=11)
            axes[0, 1].set_title(f'Misses (n={rdm_miss_enc.shape[0]})', fontsize=11)
            axes[1, 0].set_title(f'Hits (n={rdm_hit_cue.shape[0]})', fontsize=11)
            axes[1, 1].set_title(f'Misses (n={rdm_miss_cue.shape[0]})', fontsize=11)

            axes[0, 0].set_ylabel(
                f"Encoding ({SMOKE_PAIR[0]})\n"
                f"t = {feedback_times[fb_idx]:.2f}s\n\nTrials",
                fontsize=10,
            )
            axes[1, 0].set_ylabel(
                f"Retrieval ({SMOKE_PAIR[1]})\n"
                f"t = {cue_times[cue_idx]:.2f}s\n\nTrials",
                fontsize=10,
            )

            cbar = fig.colorbar(im, ax=axes, shrink=0.65, label=f'Dissimilarity\n({RDM_METHOD})')
            cbar.ax.tick_params(labelsize=9)
            fig.suptitle(
                f'Example ROI-Level RDMs for Paper\n'
                f'{example_subject} | {SMOKE_PAIR[0]}->{SMOKE_PAIR[1]}\n'
                f"timepoints selected from top-ranked channel pair: {example_selected_pair['encoding_electrode']} -> {example_selected_pair['cue_electrode']}",
                fontsize=13,
                fontweight='bold',
                y=1.02,
            )
            plt.tight_layout()
            plt.savefig(f'{RSA_GROUP_SAVE_DIR}/{example_subject}_{SMOKE_PAIR[0]}_{SMOKE_PAIR[1]}_example_rdm.pdf', dpi=300)
            plt.show()

            remembered_heat = np.asarray(example_pair_result['remembered_heatmap'], dtype=float)
            forgotten_heat = np.asarray(example_pair_result['forgotten_heatmap'], dtype=float)
            diff_heat = np.asarray(example_pair_result['diff_heatmap'], dtype=float)
            vmax_heat = np.nanmax(np.abs(np.concatenate([
                remembered_heat.ravel(),
                forgotten_heat.ravel(),
                diff_heat.ravel(),
            ])))
            if (not np.isfinite(vmax_heat)) or vmax_heat == 0:
                vmax_heat = 1.0

            fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
            heatmaps = [
                ('Hits', remembered_heat),
                ('Misses', forgotten_heat),
                ('Hits - Misses', diff_heat),
            ]
            for ax, (title, heatmap) in zip(axes, heatmaps):
                im = ax.imshow(
                    heatmap,
                    origin='lower',
                    aspect='auto',
                    extent=[float(cue_times.min()), float(cue_times.max()), float(feedback_times.min()), float(feedback_times.max())],
                    cmap='RdBu_r',
                    vmin=-vmax_heat,
                    vmax=vmax_heat,
                )
                ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
                ax.axvline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
                ax.set_title(title, fontsize=11, fontweight='bold')
                ax.set_xlabel(f'Retrieval Time ({SMOKE_PAIR[1]}, s)')
                ax.set_ylabel(f'Encoding Time ({SMOKE_PAIR[0]}, s)')
                fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)

            fig.suptitle(
                f'Example Single-Channel Cross-Temporal RSA\n'
                f"{example_subject} | {example_selected_pair['encoding_electrode']} -> {example_selected_pair['cue_electrode']}",
                fontsize=13,
                fontweight='bold',
                y=1.03,
            )
            plt.tight_layout()
            # plt.savefig(f'{RSA_GROUP_SAVE_DIR}/{example_subject}_{SMOKE_PAIR[0]}_{SMOKE_PAIR[1]}_example_rdm.pdf', dpi=300)
            plt.show()


In [ ]:
RSA_GROUP_SAVE_DIR

In [ ]:
RSA_GROUP_SAVE_DIR

## Batch RSA Pipeline

In [ ]:
# =============================
# Batch subject+pair run
# =============================
rsa_batch_results = None
if RUN_BATCH_RSA:
    rsa_batch_results = run_group_crossregion_rsa_pipeline(
        subjects=subjects,
        roi_pairs=ROI_PAIRS,
        base_dir=base_dir,
        load_dir=load_dir,
        trial_save_dir=RSA_TRIAL_SAVE_DIR,
        group_save_dir=RSA_GROUP_SAVE_DIR,
        event_encoding=EVENT_ENCODING,
        event_cue=EVENT_CUE,
        freq_range=FREQ_RANGE,
        downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS,
        min_trials=MIN_TRIALS,
        n_permutations=RSA_N_PERM,
        shuffle_type=RSA_SHUFFLE_TYPE,
        seed=RSA_SEED,
        rdm_method=RDM_METHOD,
        rdm_compare_method=RDM_COMPARE_METHOD,
        run_group_cluster=RUN_GROUP_CLUSTER,
        group_cluster_n_permutations=5000,
        group_cluster_tail=1,
        group_cluster_threshold=0,
        n_jobs_outer=N_JOBS_OUTER,
        overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
        verbose=VERBOSE,
    )

    print('Batch complete.')
    if rsa_batch_results is not None:
        print('Group rows:', len(rsa_batch_results['group_summary']))
        print(rsa_batch_results['group_summary'][['encoding_roi', 'cue_roi', 'n_subjects', 'n_sig_clusters']])


In [ ]:
RSA_GROUP_SAVE_DIR

## Group-Level Cluster Plot

In [ ]:
# load the results for ACC - HPC
rsa_batch_results = np.load(f'{RSA_GROUP_SAVE_DIR}/ACC_to_HPC_group_rsa.npz', allow_pickle=True)

In [ ]:
t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
    rsa_batch_results['observed_diffs_stack'],
    n_permutations=1000,
    tail=1,
    out_type='mask',
    seed=42,
    verbose=True,
)
sig_mask = np.zeros(rsa_batch_results['rsa_batch_results'].shape[1:], dtype=bool)
for c_mask, p in zip(clusters, cluster_pvals):
    if p <= 0.05:
        sig_mask |= c_mask.astype(bool)
cluster_res = {
    't_obs': np.asarray(t_obs),
    'clusters': clusters,
    'cluster_pvals': np.asarray(cluster_pvals),
    'sig_mask': sig_mask,
}

In [ ]:
cluster_pvals

## Z-normalized group cluster (ACC→HPC)

Group-level cluster test on the stack of **per-subject `z_diff_heatmap`** maps (observed rho difference vs permutation null), addressing aggregation of null-standardized differences across subjects.

In [ ]:
# =============================
# Z-normalized group cluster (per-subject z_diff_heatmap stack)
# One plot + meta NPZ per ROI pair in ROI_PAIRS
# =============================
import shutil

RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
Z_CLUSTER_OUT = os.path.join(RSA_CONTROLS_DIR, 'z_cluster')
os.makedirs(Z_CLUSTER_OUT, exist_ok=True)

cluster_res_z_tail1 = None
cluster_res_z_tail0 = None
sig_mask_z = None
group_mean_z = None
cluster_selection = 'none'
z_stack = None
z_subjects = None

for encoding_roi, cue_roi in ROI_PAIRS:
    pair_paths_z = build_crossregion_rsa_paths(
        trial_save_dir=RSA_TRIAL_SAVE_DIR,
        group_save_dir=RSA_GROUP_SAVE_DIR,
        encoding_roi=encoding_roi,
        cue_roi=cue_roi,
        subject=None,
    )
    pair_name = pair_paths_z['pair_name']
    z_stack, z_subjects = load_z_diff_stack_from_pair_dir(pair_paths_z['pair_dir'], subjects)

    if z_stack.shape[0] < 2:
        print(f'[{pair_name}] skip: need >=2 subjects with z_diff NPZs; got {z_stack.shape[0]}')
        continue

    group_mean_z = np.nanmean(z_stack, axis=0).astype(np.float32)

    cluster_res_z_tail1 = run_group_crossregion_rsa_cluster_test(
        z_stack, n_permutations=5000, tail=1, threshold=None, seed=42, verbose=False
    )
    cluster_res_z_tail0 = run_group_crossregion_rsa_cluster_test(
        z_stack, n_permutations=5000, tail=0, threshold=None, seed=43, verbose=False
    )

    sig_mask_z = np.asarray(cluster_res_z_tail1['sig_mask'], dtype=bool)
    cluster_selection = 'sig_p_0.05' if np.any(sig_mask_z) else 'no_sig_cluster'
    if not np.any(sig_mask_z):
        print(
            f'[{pair_name}] No cluster significant at p<=0.05 on z-diff (tail=1); '
            'leaving sig_mask_z empty (no contour / no exploratory mask).'
        )

    npz0 = os.path.join(pair_paths_z['pair_dir'], f'{z_subjects[0]}_rsa.npz')
    with np.load(npz0, allow_pickle=True) as npz:
        fb_t = np.asarray(npz['feedback_times'], dtype=float)
        cue_t = np.asarray(npz['cue_times'], dtype=float)

    meta_path = os.path.join(Z_CLUSTER_OUT, f'{pair_name}_z_cluster_meta.npz')
    np.savez_compressed(
        meta_path,
        z_stack=z_stack.astype(np.float32),
        subjects_included=np.asarray(z_subjects, dtype=object),
        group_mean_z=group_mean_z,
        sig_mask_z=sig_mask_z,
        cluster_sig_mask_tail1=np.asarray(cluster_res_z_tail1['sig_mask']),
        cluster_pvals_tail1=np.asarray(cluster_res_z_tail1['cluster_pvals']),
        cluster_pvals_tail0=np.asarray(cluster_res_z_tail0['cluster_pvals']),
        cluster_selection=np.asarray(cluster_selection),
        feedback_times=fb_t,
        cue_times=cue_t,
        encoding_roi=np.asarray(encoding_roi),
        cue_roi=np.asarray(cue_roi),
    )

    vmax = float(np.nanmax(np.abs(group_mean_z)))
    if (not np.isfinite(vmax)) or vmax == 0:
        vmax = 1.0
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(
        group_mean_z,
        origin='lower',
        aspect='auto',
        extent=[float(cue_t.min()), float(cue_t.max()), float(fb_t.min()), float(fb_t.max())],
        cmap='RdBu_r',
        vmin=-vmax,
        vmax=vmax,
    )
    if sig_mask_z is not None and sig_mask_z.size and np.any(sig_mask_z):
        ax.contour(cue_t, fb_t, sig_mask_z.astype(int), levels=[0.5], colors='k', linewidths=1.5)
    ax.set_title(f'Group mean z(diff RSA) vs permutation null\n{encoding_roi} -> {cue_roi}')
    ax.set_xlabel('Cue Time (s)')
    ax.set_ylabel('Feedback Time (s)')
    fig.colorbar(im, ax=ax, shrink=0.8, label='z (diff vs null)')
    plt.tight_layout()
    pdf_path = os.path.join(Z_CLUSTER_OUT, f'{pair_name}_z_cluster.pdf')
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
    if encoding_roi == 'ACC' and cue_roi == 'HPC':
        shutil.copyfile(pdf_path, os.path.join(Z_CLUSTER_OUT, 'ACC_HPC_z_cluster.pdf'))
        np.savez_compressed(
            os.path.join(Z_CLUSTER_OUT, 'ACC_HPC_z_cluster_meta.npz'),
            z_stack=z_stack.astype(np.float32),
            subjects_included=np.asarray(z_subjects, dtype=object),
            group_mean_z=group_mean_z,
            sig_mask_z=sig_mask_z,
            cluster_sig_mask_tail1=np.asarray(cluster_res_z_tail1['sig_mask']),
            cluster_pvals_tail1=np.asarray(cluster_res_z_tail1['cluster_pvals']),
            cluster_pvals_tail0=np.asarray(cluster_res_z_tail0['cluster_pvals']),
            cluster_selection=np.asarray(cluster_selection),
            feedback_times=fb_t,
            cue_times=cue_t,
        )
    plt.show()

    cp1 = np.asarray(cluster_res_z_tail1['cluster_pvals'])
    print(
        f'[{pair_name}] Z-cluster tail=1: n_sig_clusters {int(np.sum(cp1 <= 0.05))} | '
        f'n_clusters {len(cp1)} | n_subj {z_stack.shape[0]}'
    )

# Restore ACC->HPC workspace variables for downstream per-subject masked z / diagnostics (order-independent)
acc_hpc_meta = os.path.join(Z_CLUSTER_OUT, 'ACC_to_HPC_z_cluster_meta.npz')
if os.path.isfile(acc_hpc_meta):
    mz = np.load(acc_hpc_meta, allow_pickle=True)
    z_stack = np.asarray(mz['z_stack'], dtype=float)
    z_subjects = [str(x) for x in np.asarray(mz['subjects_included'])]
    group_mean_z = np.asarray(mz['group_mean_z'], dtype=float)
    sig_mask_z = np.asarray(mz['sig_mask_z'], dtype=bool)
    cluster_selection = str(np.asarray(mz['cluster_selection']).item()) if 'cluster_selection' in mz.files else 'none'
    print('Downstream vars loaded from ACC_to_HPC_z_cluster_meta.npz (per-subject masked z / diagnostics).')
elif os.path.isfile(os.path.join(Z_CLUSTER_OUT, 'ACC_HPC_z_cluster_meta.npz')):
    mz = np.load(os.path.join(Z_CLUSTER_OUT, 'ACC_HPC_z_cluster_meta.npz'), allow_pickle=True)
    z_stack = np.asarray(mz['z_stack'], dtype=float)
    z_subjects = [str(x) for x in np.asarray(mz['subjects_included'])]
    group_mean_z = np.asarray(mz['group_mean_z'], dtype=float)
    sig_mask_z = np.asarray(mz['sig_mask_z'], dtype=bool)
    cluster_selection = str(np.asarray(mz['cluster_selection']).item()) if 'cluster_selection' in mz.files else 'none'
    print('Downstream vars loaded from ACC_HPC_z_cluster_meta.npz.')
else:
    print('ACC->HPC meta not found; run batch RSA for ACC->HPC or check ROI_PAIRS.')


## Cross-temporal reinstatement (item-specific RI)

At each (feedback time × cue time), within hits and misses separately, build the aligned trial×trial cosine similarity **S** (same preprocessing as trial-wise RI), then **RI = mean(diag S) − mean(off-diagonal)**. Group inference stacks per-subject **RI_hit − RI_miss** maps and runs **`permutation_cluster_1samp_test`** (tail 1, threshold 0), matching the canonical RSA group-cluster style (no z-null on RI).

In [ ]:
# =============================
# Cross-temporal item-specific reinstatement (RI) heatmaps + group cluster on diff
# Per-subject NPZs under TRI_OUT; group stack uses same_cluster-style cluster test as RSA
# =============================
RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
TRI_OUT = os.path.join(RSA_CONTROLS_DIR, 'trial_reinstatement')
os.makedirs(TRI_OUT, exist_ok=True)

RI_ENCODING_ROI = 'ACC'
RI_CUE_ROI = 'HPC'

diff_rows = []
rem_rows = []
forg_rows = []
loaded_subjects = []
fb_t = None
cue_t = None

for subj in subjects:
    try:
        s_obj = Subject(subj, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)
        data = load_subject_crossregion_hfa(
            subject_obj=s_obj,
            encoding_roi=RI_ENCODING_ROI,
            cue_roi=RI_CUE_ROI,
            event_encoding=EVENT_ENCODING,
            event_cue=EVENT_CUE,
            freq_range=FREQ_RANGE,
            downsample_factor=DOWNSAMPLE_FACTOR,
            win_ms=WIN_MS,
            min_trials=MIN_TRIALS,
        )
        ri_maps = compute_cross_temporal_reinstatement_maps(
            data['encoding_hfa'],
            data['cue_hfa'],
            data['memory_outcomes'],
        )
        fb_t = np.asarray(data['feedback_times'], dtype=float)
        cue_t = np.asarray(data['cue_times'], dtype=float)

        out_sub = os.path.join(
            TRI_OUT,
            f'{subj}_cross_temporal_ri_{RI_ENCODING_ROI}_{RI_CUE_ROI}.npz',
        )
        np.savez_compressed(
            out_sub,
            remembered_heatmap=ri_maps['remembered_heatmap'],
            forgotten_heatmap=ri_maps['forgotten_heatmap'],
            diff_heatmap=ri_maps['diff_heatmap'],
            remembered_match_heatmap=ri_maps['remembered_match_heatmap'],
            remembered_mismatch_heatmap=ri_maps['remembered_mismatch_heatmap'],
            forgotten_match_heatmap=ri_maps['forgotten_match_heatmap'],
            forgotten_mismatch_heatmap=ri_maps['forgotten_mismatch_heatmap'],
            n_remembered=ri_maps['n_remembered'],
            n_forgotten=ri_maps['n_forgotten'],
            feedback_times=fb_t,
            cue_times=cue_t,
            encoding_roi=np.asarray(RI_ENCODING_ROI),
            cue_roi=np.asarray(RI_CUE_ROI),
        )

        d = np.asarray(ri_maps['diff_heatmap'], dtype=np.float64)
        r = np.asarray(ri_maps['remembered_heatmap'], dtype=np.float64)
        f = np.asarray(ri_maps['forgotten_heatmap'], dtype=np.float64)
        diff_rows.append(np.where(np.isfinite(d), d, 0.0))
        rem_rows.append(np.where(np.isfinite(r), r, 0.0))
        forg_rows.append(np.where(np.isfinite(f), f, 0.0))
        loaded_subjects.append(subj)
    except Exception as e:
        print(f'skip {subj}: {e}')

if not loaded_subjects or fb_t is None:
    raise RuntimeError('No subjects produced cross-temporal RI maps; check data paths and MIN_TRIALS.')

diff_stack = np.stack(diff_rows, axis=0).astype(np.float64)
rem_stack = np.stack(rem_rows, axis=0).astype(np.float64)
forg_stack = np.stack(forg_rows, axis=0).astype(np.float64)

group_mean_rem = np.nanmean(rem_stack, axis=0).astype(np.float32)
group_mean_forg = np.nanmean(forg_stack, axis=0).astype(np.float32)
group_mean_diff = np.nanmean(diff_stack, axis=0).astype(np.float32)

ri_cluster_res = None
sig_mask_ri = np.zeros(group_mean_diff.shape, dtype=bool)
if diff_stack.shape[0] < 2:
    print(f'Only {diff_stack.shape[0]} subject(s); skipping group cluster (need >=2).')
else:
    ri_cluster_res = run_group_crossregion_rsa_cluster_test(
        diff_stack,
        n_permutations=5000,
        tail=1,
        threshold=0,
        seed=44,
        verbose=False,
    )
    sig_mask_ri = np.asarray(ri_cluster_res['sig_mask'], dtype=bool)
    if not np.any(sig_mask_ri):
        print(
            f'[{RI_ENCODING_ROI}->{RI_CUE_ROI}] No cluster significant at p<=0.05 on RI diff '
            '(tail=1, threshold=0); leaving sig_mask_ri empty.'
        )

group_npz = os.path.join(TRI_OUT, f'group_ri_{RI_ENCODING_ROI}_{RI_CUE_ROI}.npz')
save_kw = dict(
    diff_stack=diff_stack.astype(np.float32),
    remembered_stack=rem_stack.astype(np.float32),
    forgotten_stack=forg_stack.astype(np.float32),
    subjects_included=np.asarray(loaded_subjects, dtype=object),
    group_mean_remembered=group_mean_rem,
    group_mean_forgotten=group_mean_forg,
    group_mean_diff=group_mean_diff,
    sig_mask_ri=sig_mask_ri,
    feedback_times=fb_t,
    cue_times=cue_t,
    encoding_roi=np.asarray(RI_ENCODING_ROI),
    cue_roi=np.asarray(RI_CUE_ROI),
)
if ri_cluster_res is not None:
    save_kw['cluster_pvals'] = np.asarray(ri_cluster_res['cluster_pvals'])
    save_kw['t_obs'] = np.asarray(ri_cluster_res['t_obs'])
else:
    save_kw['cluster_pvals'] = np.array([])
    save_kw['t_obs'] = np.array([])

np.savez_compressed(group_npz, **save_kw)

vmax_heat = float(
    np.nanmax(
        np.abs(
            np.concatenate(
                [
                    group_mean_rem.ravel(),
                    group_mean_forg.ravel(),
                    group_mean_diff.ravel(),
                ]
            )
        )
    )
)
if (not np.isfinite(vmax_heat)) or vmax_heat == 0:
    vmax_heat = 1.0

extent = [
    float(cue_t.min()),
    float(cue_t.max()),
    float(fb_t.min()),
    float(fb_t.max()),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
heatmaps = [
    ('Hits (RI)', group_mean_rem),
    ('Misses (RI)', group_mean_forg),
    ('Hits − Misses (RI)', group_mean_diff),
]
for ax, (title, heatmap) in zip(axes, heatmaps):
    im = ax.imshow(
        heatmap,
        origin='lower',
        aspect='auto',
        extent=extent,
        cmap='RdBu_r',
        vmin=-vmax_heat,
        vmax=vmax_heat,
    )
    if title.startswith('Hits −') and sig_mask_ri.size and np.any(sig_mask_ri):
        ax.contour(
            cue_t,
            fb_t,
            sig_mask_ri.astype(int),
            levels=[0.5],
            colors='k',
            linewidths=1.5,
        )
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(f'Cue time ({RI_CUE_ROI}, s)')
    ax.set_ylabel(f'Feedback time ({RI_ENCODING_ROI}, s)')
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)

fig.suptitle(
    f'Group mean cross-temporal reinstatement | {RI_ENCODING_ROI}→{RI_CUE_ROI} | n={len(loaded_subjects)}',
    fontsize=12,
    fontweight='bold',
    y=1.02,
)
plt.tight_layout()
pdf_path = os.path.join(TRI_OUT, f'group_ri_{RI_ENCODING_ROI}_{RI_CUE_ROI}.pdf')
plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
plt.show()

cp = np.asarray(ri_cluster_res['cluster_pvals']) if ri_cluster_res is not None else np.array([])
print(
    f'RI diff cluster test: n_subj={len(loaded_subjects)} | '
    f"n_sig_clusters={int(np.sum(cp <= 0.05))} | n_clusters={len(cp)} | saved {group_npz}"
)


In [ ]:
# Inspect saved group cross-temporal RI bundle (after cell above)
_tri = os.path.join('analysis_outputs', 'rsa_controls', 'trial_reinstatement', 'group_ri_ACC_HPC.npz')
if os.path.isfile(_tri):
    _g = np.load(_tri, allow_pickle=True)
    print('Keys:', sorted(_g.files))
else:
    print('Not found:', _tri)



In [ ]:
# Trial-level merged_df / mixed-effects on per-trial RI removed.
# Group inference for RI is the cluster test on stacked per-subject diff maps (previous cell).
pass



In [ ]:
ROI_PAIRS

In [ ]:
# Per-trial reinstatement DataFrame removed; use group_ri_ACC_HPC.npz and per-subject *_cross_temporal_ri_*.npz
pass



In [ ]:
# Barplot of trial RI by RPE bin removed with trial-level table.
pass



In [ ]:
# Per-trial reinstatement DataFrame removed; use group_ri_ACC_HPC.npz and per-subject *_cross_temporal_ri_*.npz
pass



## Model RSA (partial RPE category) and RI category decomposition

Parsimonious follow-up: **rho_full** = cross-region RSA; **rho_item** = partial RSA controlling nominal RPE-category RDM; **ri_cat** / **ri_item** decompose trial-aligned cosine similarity **S** into category structure vs item identity beyond category. Per-subject maps are saved under `analysis_outputs/rsa_controls/model_rsa_partial/` and `ri_category_decomp/`; ACC→HPC extensive figures under `acc_hpc_reactivation_content/`.


In [ ]:
# =============================
# Cells A+B: per-subject partial RSA + RI category maps (all ROI_PAIRS)
# =============================
RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
PARTIAL_OUT = os.path.join(RSA_CONTROLS_DIR, 'model_rsa_partial')
RI_CAT_OUT = os.path.join(RSA_CONTROLS_DIR, 'ri_category_decomp')
os.makedirs(PARTIAL_OUT, exist_ok=True)
os.makedirs(RI_CAT_OUT, exist_ok=True)

RUN_RPE_FOLLOWUP = True
rpe_followup_errors = []

if RUN_RPE_FOLLOWUP:
    for encoding_roi, cue_roi in ROI_PAIRS:
        pair_name = f"{encoding_roi}_to_{cue_roi}"
        partial_pair_dir = os.path.join(PARTIAL_OUT, pair_name)
        ri_pair_dir = os.path.join(RI_CAT_OUT, pair_name)
        os.makedirs(partial_pair_dir, exist_ok=True)
        os.makedirs(ri_pair_dir, exist_ok=True)
        for subj in subjects:
            try:
                s_obj = Subject(subj, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)
                data = load_subject_crossregion_hfa(
                    subject_obj=s_obj,
                    encoding_roi=encoding_roi,
                    cue_roi=cue_roi,
                    event_encoding=EVENT_ENCODING,
                    event_cue=EVENT_CUE,
                    freq_range=FREQ_RANGE,
                    downsample_factor=DOWNSAMPLE_FACTOR,
                    win_ms=WIN_MS,
                    min_trials=MIN_TRIALS,
                )
                rpe_cat = np.asarray(data['rpe_categories'], dtype=object)
                if np.all(pd.isna(rpe_cat)):
                    raise ValueError('missing rpe_cat on combined_df for this subject')

                partial = compute_cross_temporal_rsa_partial_from_hfa(
                    data['encoding_hfa'],
                    data['cue_hfa'],
                    data['memory_outcomes'],
                    rpe_cat,
                    rdm_method=RDM_METHOD,
                    rdm_compare_method=RDM_COMPARE_METHOD,
                )
                ri_cd = compute_cross_temporal_reinstatement_category_maps(
                    data['encoding_hfa'],
                    data['cue_hfa'],
                    data['memory_outcomes'],
                    rpe_cat,
                )
                fb_t = np.asarray(data['feedback_times'], dtype=float)
                cue_t = np.asarray(data['cue_times'], dtype=float)

                np.savez_compressed(
                    os.path.join(partial_pair_dir, f'{subj}_rsa_partial_{encoding_roi}_{cue_roi}.npz'),
                    rho_full_remembered=partial['rho_full_remembered'],
                    rho_full_forgotten=partial['rho_full_forgotten'],
                    rho_full_diff=partial['rho_full_diff'],
                    rho_item_remembered=partial['rho_item_remembered'],
                    rho_item_forgotten=partial['rho_item_forgotten'],
                    rho_item_diff=partial['rho_item_diff'],
                    rho_cat_acc_remembered=partial['rho_cat_acc_remembered'],
                    rho_cat_acc_forgotten=partial['rho_cat_acc_forgotten'],
                    rho_cat_acc_diff=partial['rho_cat_acc_diff'],
                    rho_cat_cue_remembered=partial['rho_cat_cue_remembered'],
                    rho_cat_cue_forgotten=partial['rho_cat_cue_forgotten'],
                    rho_cat_cue_diff=partial['rho_cat_cue_diff'],
                    feedback_times=fb_t,
                    cue_times=cue_t,
                    rpe_categories=np.asarray(rpe_cat, dtype=object),
                    encoding_roi=np.asarray(encoding_roi),
                    cue_roi=np.asarray(cue_roi),
                    subject=np.asarray(subj),
                )
                np.savez_compressed(
                    os.path.join(ri_pair_dir, f'{subj}_ri_catdecomp_{encoding_roi}_{cue_roi}.npz'),
                    ri_item_remembered=ri_cd['ri_item_remembered'],
                    ri_item_forgotten=ri_cd['ri_item_forgotten'],
                    ri_item_diff=ri_cd['ri_item_diff'],
                    ri_cat_remembered=ri_cd['ri_cat_remembered'],
                    ri_cat_forgotten=ri_cd['ri_cat_forgotten'],
                    ri_cat_diff=ri_cd['ri_cat_diff'],
                    feedback_times=fb_t,
                    cue_times=cue_t,
                    rpe_categories=np.asarray(rpe_cat, dtype=object),
                    encoding_roi=np.asarray(encoding_roi),
                    cue_roi=np.asarray(cue_roi),
                    subject=np.asarray(subj),
                )
            except Exception as e:
                rpe_followup_errors.append((pair_name, subj, repr(e)))
    print('RPE follow-up per-subject save complete. Errors:', len(rpe_followup_errors))
    if rpe_followup_errors:
        for row in rpe_followup_errors[:20]:
            print(row)
        if len(rpe_followup_errors) > 20:
            print('...')
else:
    print('Set RUN_RPE_FOLLOWUP = True to compute partial RSA + RI category maps.')


In [ ]:
# =============================
# Cell C: group cluster tests + PDFs per ROI pair (rho_full/item/ri_cat/ri_item diff)
# =============================
RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
PARTIAL_OUT = os.path.join(RSA_CONTROLS_DIR, 'model_rsa_partial')
RI_CAT_OUT = os.path.join(RSA_CONTROLS_DIR, 'ri_category_decomp')
GROUP_CLUSTER_N = 5000


def _load_metric_stack_from_pair_dir(pair_dir, subjects, key):
    rows = []
    loaded = []
    fb_t = None
    cue_t = None
    for subj in subjects:
        pat = os.path.join(pair_dir, f'{subj}_rsa_partial_*.npz')
        files = glob.glob(pat)
        if len(files) != 1:
            continue
        path = files[0]
        with np.load(path, allow_pickle=True) as npz:
            if key not in npz.files:
                continue
            z = np.asarray(npz[key], dtype=np.float64)
            z = np.where(np.isfinite(z), z, 0.0)
            rows.append(z)
            loaded.append(subj)
            if fb_t is None and 'feedback_times' in npz.files:
                fb_t = np.asarray(npz['feedback_times'], dtype=float)
                cue_t = np.asarray(npz['cue_times'], dtype=float)
    if len(rows) == 0:
        return np.empty((0, 0, 0), dtype=np.float64), [], None, None
    return np.stack(rows, axis=0).astype(np.float64), loaded, fb_t, cue_t


def _load_ri_metric_stack(pair_dir, subjects, key):
    rows = []
    loaded = []
    fb_t = None
    cue_t = None
    for subj in subjects:
        pat = os.path.join(pair_dir, f'{subj}_ri_catdecomp_*.npz')
        files = glob.glob(pat)
        if len(files) != 1:
            continue
        path = files[0]
        with np.load(path, allow_pickle=True) as npz:
            if key not in npz.files:
                continue
            z = np.asarray(npz[key], dtype=np.float64)
            z = np.where(np.isfinite(z), z, 0.0)
            rows.append(z)
            loaded.append(subj)
            if fb_t is None and 'feedback_times' in npz.files:
                fb_t = np.asarray(npz['feedback_times'], dtype=float)
                cue_t = np.asarray(npz['cue_times'], dtype=float)
    if len(rows) == 0:
        return np.empty((0, 0, 0), dtype=np.float64), [], None, None
    return np.stack(rows, axis=0).astype(np.float64), loaded, fb_t, cue_t


if RUN_RPE_FOLLOWUP or os.path.isdir(PARTIAL_OUT):
    for encoding_roi, cue_roi in ROI_PAIRS:
        pair_name = f"{encoding_roi}_to_{cue_roi}"
        partial_pair_dir = os.path.join(PARTIAL_OUT, pair_name)
        ri_pair_dir = os.path.join(RI_CAT_OUT, pair_name)
        if not os.path.isdir(partial_pair_dir):
            continue

        stacks = {}
        for k in ('rho_full_diff', 'rho_item_diff'):
            stacks[k], subj_p, fb_t, cue_t = _load_metric_stack_from_pair_dir(
                partial_pair_dir, subjects, k,
            )
        for k in ('ri_cat_diff', 'ri_item_diff'):
            stacks[k], subj_r, fb2, cue2 = _load_ri_metric_stack(
                ri_pair_dir, subjects, k,
            )
            if fb_t is None:
                fb_t, cue_t = fb2, cue2

        rf = stacks.get('rho_full_diff')
        if rf is None or rf.shape[0] < 2:
            print(f'[{pair_name}] skip group cluster: need >=2 subjects with partial NPZs')
            continue

        group_bundle = {}
        for metric, D in stacks.items():
            if D is None or D.shape[0] < 2:
                continue
            group_bundle[metric] = run_group_crossregion_rsa_cluster_test(
                D,
                n_permutations=GROUP_CLUSTER_N,
                tail=1,
                threshold=0,
                seed=45 + abs(hash(metric)) % 1000,
                verbose=False,
            )

        extent = [
            float(cue_t.min()), float(cue_t.max()),
            float(fb_t.min()), float(fb_t.max()),
        ]
        for metric, D in stacks.items():
            if D is None or D.shape[0] < 2 or metric not in group_bundle:
                continue
            res = group_bundle[metric]
            gmean = np.nanmean(D, axis=0).astype(np.float32)
            sig = np.asarray(res['sig_mask'], dtype=bool)
            vmax = float(np.nanmax(np.abs(gmean)))
            if not np.isfinite(vmax) or vmax == 0:
                vmax = 1.0
            fig, ax = plt.subplots(figsize=(6, 5))
            im = ax.imshow(
                gmean,
                origin='lower',
                aspect='auto',
                extent=extent,
                cmap='RdBu_r',
                vmin=-vmax,
                vmax=vmax,
            )
            if sig.size and np.any(sig):
                ax.contour(
                    cue_t, fb_t, sig.astype(int),
                    levels=[0.5], colors='k', linewidths=1.5,
                )
            ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
            ax.axvline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
            ax.set_xlabel('Cue time (s)')
            ax.set_ylabel('Feedback time (s)')
            ax.set_title(f'Group mean {metric}\n{pair_name} n={D.shape[0]}')
            fig.colorbar(im, ax=ax, shrink=0.8)
            plt.tight_layout()
            outp = os.path.join(partial_pair_dir, f'{pair_name}_{metric}_group.pdf')
            plt.savefig(outp, dpi=300, bbox_inches='tight')
            plt.show()

        np.savez_compressed(
            os.path.join(partial_pair_dir, f'{pair_name}_model_rsa_group_meta.npz'),
            rho_full_diff_stack=stacks.get('rho_full_diff', np.array([])),
            rho_item_diff_stack=stacks.get('rho_item_diff', np.array([])),
            subjects_included=np.asarray(subj_p, dtype=object),
            feedback_times=fb_t,
            cue_times=cue_t,
            encoding_roi=np.asarray(encoding_roi),
            cue_roi=np.asarray(cue_roi),
        )
        np.savez_compressed(
            os.path.join(ri_pair_dir, f'{pair_name}_ri_catdecomp_group_meta.npz'),
            ri_cat_diff_stack=stacks.get('ri_cat_diff', np.array([])),
            ri_item_diff_stack=stacks.get('ri_item_diff', np.array([])),
            subjects_included=np.asarray(subj_r, dtype=object),
            feedback_times=fb_t,
            cue_times=cue_t,
            encoding_roi=np.asarray(encoding_roi),
            cue_roi=np.asarray(cue_roi),
        )
        print(f'[{pair_name}] saved group meta + cluster PDFs under partial/ri_catdecomp dirs')
else:
    print('Skip Cell C: no PARTIAL_OUT directory')


In [ ]:
# =============================
# Cell D: masked scalars + simple bar (ACC -> HPC only, sig_mask_z)
# =============================
RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
PARTIAL_OUT = os.path.join(RSA_CONTROLS_DIR, 'model_rsa_partial')
RI_CAT_OUT = os.path.join(RSA_CONTROLS_DIR, 'ri_category_decomp')
MASKED_PARTIAL_CSV = os.path.join(PARTIAL_OUT, 'ACC_to_HPC', 'per_subject_masked_acc_hpc.csv')
MASKED_PARTIAL_PDF = os.path.join(PARTIAL_OUT, 'ACC_to_HPC', 'per_subject_masked_partial_acc_hpc.pdf')

acc_hpc_partial_dir = os.path.join(PARTIAL_OUT, 'ACC_to_HPC')
meta_z = os.path.join(RSA_CONTROLS_DIR, 'z_cluster', 'ACC_to_HPC_z_cluster_meta.npz')
if not os.path.isfile(meta_z):
    meta_z = os.path.join(RSA_CONTROLS_DIR, 'z_cluster', 'ACC_HPC_z_cluster_meta.npz')

if os.path.isfile(meta_z) and os.path.isdir(acc_hpc_partial_dir):
    mz = np.load(meta_z, allow_pickle=True)
    sig_mask_z_acc = np.asarray(mz['sig_mask_z'], dtype=bool)
    z_subjects_meta = [str(x) for x in np.asarray(mz['subjects_included'])]

    rows = []
    for subj in subjects:
        p_partial = os.path.join(
            acc_hpc_partial_dir, f'{subj}_rsa_partial_ACC_HPC.npz',
        )
        p_ri = os.path.join(
            RI_CAT_OUT, 'ACC_to_HPC', f'{subj}_ri_catdecomp_ACC_HPC.npz',
        )
        if not os.path.isfile(p_partial):
            continue
        with np.load(p_partial, allow_pickle=True) as npz:
            rf = np.asarray(npz['rho_full_diff'], dtype=float)
            ri = np.asarray(npz['rho_item_diff'], dtype=float)
        if os.path.isfile(p_ri):
            with np.load(p_ri, allow_pickle=True) as npz:
                ric = np.asarray(npz['ri_cat_diff'], dtype=float)
                rii = np.asarray(npz['ri_item_diff'], dtype=float)
        else:
            ric = np.full_like(rf, np.nan)
            rii = np.full_like(rf, np.nan)

        m = sig_mask_z_acc
        if m.shape != rf.shape:
            print('sig_mask_z shape mismatch; skip masked scalars')
            rows = []
            break
        if not np.any(m):
            print('empty sig_mask_z; skip masked scalars')
            rows = []
            break

        rows.append({
            'participant': subj,
            'masked_rho_full_diff': float(np.nanmean(rf[m])),
            'masked_rho_item_diff': float(np.nanmean(ri[m])),
            'masked_ri_cat_diff': float(np.nanmean(ric[m])) if np.any(np.isfinite(ric)) else np.nan,
            'masked_ri_item_diff': float(np.nanmean(rii[m])) if np.any(np.isfinite(rii)) else np.nan,
        })

    if rows:
        df_mask = pd.DataFrame(rows)
        os.makedirs(os.path.dirname(MASKED_PARTIAL_CSV), exist_ok=True)
        df_mask.to_csv(MASKED_PARTIAL_CSV, index=False)

        x = np.arange(len(df_mask))
        fig, ax = plt.subplots(figsize=(max(8.0, 0.42 * len(df_mask)), 4.5))
        w = 0.35
        ax.bar(x - w / 2, df_mask['masked_rho_full_diff'], width=w, label='rho_full_diff', color='steelblue', alpha=0.85, edgecolor='k', linewidth=0.5)
        ax.bar(x + w / 2, df_mask['masked_rho_item_diff'], width=w, label='rho_item_diff', color='coral', alpha=0.85, edgecolor='k', linewidth=0.5)
        ax.axhline(0.0, color='k', linestyle='-', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(df_mask['participant'], rotation=45, ha='right')
        ax.set_ylabel('mean map inside sig_mask_z')
        ax.set_title('ACC→HPC masked partial RSA (z-cluster mask)')
        ax.legend()
        plt.tight_layout()
        plt.savefig(MASKED_PARTIAL_PDF, dpi=300, bbox_inches='tight')
        plt.show()

        a = df_mask['masked_rho_full_diff'].to_numpy(dtype=float)
        b = df_mask['masked_rho_item_diff'].to_numpy(dtype=float)
        if len(a) >= 5:
            try:
                wstat, p_ab = wilcoxon(a, b, alternative='two-sided')
            except Exception:
                p_ab = np.nan
            print('Wilcoxon rho_full_diff vs rho_item_diff (paired): p =', p_ab)
        for col, lab in [
            ('masked_rho_full_diff', 'rho_full'),
            ('masked_rho_item_diff', 'rho_item'),
            ('masked_ri_cat_diff', 'ri_cat'),
            ('masked_ri_item_diff', 'ri_item'),
        ]:
            v = df_mask[col].dropna().to_numpy(dtype=float)
            if len(v) >= 5:
                try:
                    w0, p0 = wilcoxon(v, alternative='two-sided')
                except Exception:
                    p0 = np.nan
                print(f'Wilcoxon {lab} vs 0: p =', p0)
        print('Saved', MASKED_PARTIAL_CSV, MASKED_PARTIAL_PDF)
else:
    print('Skip Cell D: need z_cluster meta + ACC_to_HPC partial NPZs')


In [ ]:
# =============================
# Cell E: ACC→HPC extensive plot pack (figures 1–8 + combined PDF)
# =============================
from scipy.spatial.distance import squareform

try:
    RSA_CONTROLS_DIR
except NameError:
    RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
try:
    PARTIAL_OUT
except NameError:
    PARTIAL_OUT = os.path.join(RSA_CONTROLS_DIR, 'model_rsa_partial')
try:
    RI_CAT_OUT
except NameError:
    RI_CAT_OUT = os.path.join(RSA_CONTROLS_DIR, 'ri_category_decomp')

ACC_HPC_PLOT_DIR = os.path.join(RSA_CONTROLS_DIR, 'acc_hpc_reactivation_content')
os.makedirs(ACC_HPC_PLOT_DIR, exist_ok=True)

ENC_ROI, CUE_ROI = 'ACC', 'HPC'
pair_tag = f'{ENC_ROI}_to_{CUE_ROI}'
partial_dir = os.path.join(PARTIAL_OUT, pair_tag)
ri_dir = os.path.join(RI_CAT_OUT, pair_tag)
meta_z_path = os.path.join(RSA_CONTROLS_DIR, 'z_cluster', 'ACC_to_HPC_z_cluster_meta.npz')
if not os.path.isfile(meta_z_path):
    meta_z_path = os.path.join(RSA_CONTROLS_DIR, 'z_cluster', 'ACC_HPC_z_cluster_meta.npz')


def _stack_npz(pair_dir, subjects, fname_fmt, key):
    """fname_fmt uses {subj}, e.g. '{subj}_rsa_partial_ACC_HPC.npz'."""
    rows = []
    loaded = []
    fb_t = cue_t = None
    for subj in subjects:
        path = os.path.join(pair_dir, fname_fmt.format(subj=subj))
        if not os.path.isfile(path):
            continue
        with np.load(path, allow_pickle=True) as npz:
            if key not in npz.files:
                continue
            a = np.asarray(npz[key], dtype=np.float64)
            a = np.where(np.isfinite(a), a, np.nan)
            rows.append(a)
            loaded.append(subj)
            if fb_t is None:
                fb_t = np.asarray(npz['feedback_times'], dtype=float)
                cue_t = np.asarray(npz['cue_times'], dtype=float)
    if not rows:
        return None, [], None, None
    return np.stack(rows, axis=0), loaded, fb_t, cue_t


if not (os.path.isdir(partial_dir) and os.path.isdir(ri_dir) and os.path.isfile(meta_z_path)):
    print('Cell E skipped: need partial_dir, ri_dir, and z_cluster meta')
else:
    mz = np.load(meta_z_path, allow_pickle=True)
    sig_mask_z = np.asarray(mz['sig_mask_z'], dtype=bool)

    fmt_p = '{subj}_rsa_partial_ACC_HPC.npz'
    fmt_r = '{subj}_ri_catdecomp_ACC_HPC.npz'

    stacks = {}
    for k in (
        'rho_full_remembered', 'rho_full_forgotten', 'rho_full_diff',
        'rho_item_remembered', 'rho_item_forgotten', 'rho_item_diff',
        'rho_cat_acc_remembered', 'rho_cat_acc_forgotten', 'rho_cat_acc_diff',
        'rho_cat_cue_remembered', 'rho_cat_cue_forgotten', 'rho_cat_cue_diff',
    ):
        stacks[k], loaded_p, fb_t, cue_t = _stack_npz(partial_dir, subjects, fmt_p, k)
    for k in (
        'ri_cat_remembered', 'ri_cat_forgotten', 'ri_cat_diff',
        'ri_item_remembered', 'ri_item_forgotten', 'ri_item_diff',
    ):
        stacks[k], loaded_r, fb2, cue2 = _stack_npz(ri_dir, subjects, fmt_r, k)
        if fb_t is None and fb2 is not None:
            fb_t, cue_t = fb2, cue2

    if stacks.get('rho_full_diff') is None or stacks['rho_full_diff'].shape[0] < 2:
        print('Cell E: insufficient stacks')
    else:
        extent = [
            float(cue_t.min()), float(cue_t.max()),
            float(fb_t.min()), float(fb_t.max()),
        ]  # imshow extent: x = cue, y = feedback
        SHARED_CBAR_MIN, SHARED_CBAR_MAX = -4.0, 4.0

        diff_names = ('rho_full_diff', 'rho_item_diff', 'ri_cat_diff', 'ri_item_diff')
        cluster_info = {}
        for name in diff_names:
            D = stacks[name]
            if D is None or D.shape[0] < 2:
                cluster_info[name] = None
                continue
            cluster_info[name] = run_group_crossregion_rsa_cluster_test(
                D, n_permutations=5000, tail=1, threshold=None,
                seed=200 + hash(name) % 800, verbose=False,
            )

        def _vmax99_list(arrs):
            parts = [a.ravel() for a in arrs if a is not None]
            if not parts:
                return 1.0
            v = np.abs(np.concatenate(parts))
            v = v[np.isfinite(v)]
            return float(np.percentile(v, 99)) if v.size else 1.0

        # ----- Figure 1: 4 x 3
        row_specs = [
            ('rho_full', 'rho_full_remembered', 'rho_full_forgotten', 'rho_full_diff'),
            ('rho_item', 'rho_item_remembered', 'rho_item_forgotten', 'rho_item_diff'),
            ('ri_cat', 'ri_cat_remembered', 'ri_cat_forgotten', 'ri_cat_diff'),
            ('ri_item', 'ri_item_remembered', 'ri_item_forgotten', 'ri_item_diff'),
        ]
        all_m = []
        all_t = []
        for _, kr, kf, kd in row_specs:
            for kk in (kr, kf):
                if stacks.get(kk) is not None:
                    all_m.append(np.nanmean(stacks[kk], axis=0))
            kd_cluster = cluster_info.get(kd)
            if kd_cluster is not None and kd_cluster.get('t_obs') is not None:
                all_t.append(np.asarray(kd_cluster['t_obs'], dtype=float))
        vmax1 = max(_vmax99_list(all_m + all_t), 1e-6)

        fig1, axes1 = plt.subplots(4, 3, figsize=(14, 16))
        for ri, (label, kr, kf, kd) in enumerate(row_specs):
            kd_cluster = cluster_info.get(kd)
            sig_k = None
            t_k = None
            if kd_cluster is not None:
                if kd_cluster.get('sig_mask') is not None:
                    sig_k = np.asarray(kd_cluster['sig_mask'], dtype=bool)
                if kd_cluster.get('t_obs') is not None:
                    t_k = np.asarray(kd_cluster['t_obs'], dtype=float)
            for ci, (k, title_suffix) in enumerate([(kr, 'Hits'), (kf, 'Misses'), (kd, 'Hits − Misses (t)')]):
                ax = axes1[ri, ci]
                if title_suffix == 'Hits − Misses (t)' and t_k is not None:
                    to_plot = t_k
                else:
                    to_plot = np.nanmean(stacks[k], axis=0)
                ax.imshow(
                    to_plot, origin='lower', aspect='auto', extent=extent,
                    cmap='RdBu_r', vmin=SHARED_CBAR_MIN, vmax=SHARED_CBAR_MAX,
                )
                if title_suffix == 'Hits − Misses (t)' and sig_k is not None and sig_k.shape == to_plot.shape and np.any(sig_k):
                    ax.contour(
                        cue_t, fb_t, sig_k.astype(int),
                        levels=[0.5], colors='k', linewidths=1.2,
                    )
                ax.axhline(0, color='k', linestyle='--', linewidth=0.4, alpha=0.5)
                ax.axvline(0, color='k', linestyle='--', linewidth=0.4, alpha=0.5)
                ax.set_title(f'{label} | {title_suffix}', fontsize=9)
                if ci == 0:
                    ax.set_ylabel(f'Feedback ({ENC_ROI})')
                if ri == 3:
                    ax.set_xlabel(f'Cue ({CUE_ROI})')
        fig1.suptitle(
            f'ACC→HPC | n={stacks["rho_full_diff"].shape[0]} | diff column: cluster p<=0.05',
            fontsize=11, y=1.01,
        )
        plt.tight_layout()
        fig1.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig1_acc_hpc_group_grid.pdf'), dpi=300, bbox_inches='tight')
        fig1.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig1_acc_hpc_group_grid.png'), dpi=300, bbox_inches='tight')
        plt.show()

        # ----- Figure 2: category encoding 2 x 3
        cat_acc_keys = (
            'rho_cat_acc_remembered', 'rho_cat_acc_forgotten', 'rho_cat_acc_diff',
        )
        cat_cue_keys = (
            'rho_cat_cue_remembered', 'rho_cat_cue_forgotten', 'rho_cat_cue_diff',
        )
        all_cat = [np.nanmean(stacks[k], axis=0) for k in cat_acc_keys + cat_cue_keys if stacks.get(k) is not None]
        vmax2 = max(_vmax99_list(all_cat), 1e-6)

        fig2, axes2 = plt.subplots(2, 3, figsize=(14, 7))
        for ci, k in enumerate(cat_acc_keys):
            gmean = np.nanmean(stacks[k], axis=0)
            axes2[0, ci].imshow(
                gmean, origin='lower', aspect='auto', extent=extent,
                cmap='RdBu_r', vmin=SHARED_CBAR_MIN, vmax=SHARED_CBAR_MAX,
            )
            axes2[0, ci].set_title(f'ACC vs M_rpe | {k.split("_")[-1]}', fontsize=9)
        for ci, k in enumerate(cat_cue_keys):
            gmean = np.nanmean(stacks[k], axis=0)
            axes2[1, ci].imshow(
                gmean, origin='lower', aspect='auto', extent=extent,
                cmap='RdBu_r', vmin=SHARED_CBAR_MIN, vmax=SHARED_CBAR_MAX,
            )
            axes2[1, ci].set_title(f'HPC vs M_rpe | {k.split("_")[-1]}', fontsize=9)
        for ax in axes2.ravel():
            ax.axhline(0, color='k', linestyle='--', linewidth=0.4, alpha=0.5)
            ax.axvline(0, color='k', linestyle='--', linewidth=0.4, alpha=0.5)
        plt.tight_layout()
        fig2.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig2_acc_hpc_category_encoding.pdf'), dpi=300, bbox_inches='tight')
        fig2.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig2_acc_hpc_category_encoding.png'), dpi=300, bbox_inches='tight')
        plt.show()

        # ----- Figures 3–4, 8: masked CSV from Cell D
        dfm_path = os.path.join(partial_dir, 'per_subject_masked_acc_hpc.csv')
        if os.path.isfile(dfm_path):
            dfm = pd.read_csv(dfm_path)
            a = dfm['masked_rho_full_diff'].to_numpy(dtype=float)
            b = dfm['masked_rho_item_diff'].to_numpy(dtype=float)

            fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))
            xx = np.arange(len(dfm))
            axes3[0].bar(xx - 0.2, a, 0.4, label='rho_full', color='steelblue', alpha=0.85)
            axes3[0].bar(xx + 0.2, b, 0.4, label='rho_item', color='coral', alpha=0.85)
            for i in range(len(dfm)):
                axes3[0].plot([i - 0.2, i + 0.2], [a[i], b[i]], 'k-', linewidth=0.5, alpha=0.5)
            axes3[0].axhline(0, color='k', linewidth=0.8)
            axes3[0].set_xticks(xx)
            axes3[0].set_xticklabels(dfm['participant'], rotation=45, ha='right')
            try:
                _, p_pair = wilcoxon(a, b, alternative='two-sided')
            except Exception:
                p_pair = np.nan
            axes3[0].set_title(f'Wilcoxon rho_full vs rho_item p={p_pair:.4g}')
            axes3[1].scatter(a, b, s=45, edgecolors='k')
            lim = max(np.nanmax(np.abs(a)), np.nanmax(np.abs(b)), 1e-6)
            axes3[1].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.6)
            axes3[1].set_xlabel('masked rho_full_diff')
            axes3[1].set_ylabel('masked rho_item_diff')
            plt.tight_layout()
            fig3.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig3_acc_hpc_variance_partition.pdf'), dpi=300, bbox_inches='tight')
            fig3.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig3_acc_hpc_variance_partition.png'), dpi=300, bbox_inches='tight')
            plt.show()

            cat_share = float(np.nanmean(a - b))
            item_part = float(np.nanmean(b))
            fig3b, axb = plt.subplots(figsize=(4, 4))
            axb.bar([0], [item_part], label='rho_item', color='coral')
            axb.bar([0], [cat_share], bottom=item_part, label='full−item', color='steelblue')
            axb.axhline(float(np.nanmean(a)), color='k', linestyle='--')
            axb.set_xticks([0])
            axb.set_xticklabels(['group'])
            axb.legend(fontsize=8)
            plt.tight_layout()
            fig3b.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig3b_acc_hpc_stacked_decomp.pdf'), dpi=300, bbox_inches='tight')
            fig3b.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig3b_acc_hpc_stacked_decomp.png'), dpi=300, bbox_inches='tight')
            plt.show()

            metrics = [
                ('masked_rho_full_diff', 'rho_full_diff'),
                ('masked_rho_item_diff', 'rho_item_diff'),
                ('masked_ri_cat_diff', 'ri_cat_diff'),
                ('masked_ri_item_diff', 'ri_item_diff'),
            ]
            fig4, axes4 = plt.subplots(2, 2, figsize=(12, 8))
            for ax, (col, name) in zip(axes4.ravel(), metrics):
                v = dfm[col].dropna().to_numpy(dtype=float)
                if len(v) == 0:
                    continue
                order = np.argsort(-v)
                cols = ['steelblue' if val >= 0 else 'salmon' for val in v[order]]
                ax.bar(np.arange(len(v)), v[order], color=cols, edgecolor='k', linewidth=0.4)
                ax.axhline(0, color='k', linewidth=0.8)
                ax.axhline(np.nanmean(v), color='coral', linestyle='--')
                ax.set_xticks(np.arange(len(v)))
                ax.set_xticklabels(np.asarray(dfm['participant'])[order], rotation=45, ha='right')
                ax.set_title(name)
                try:
                    _, p0 = wilcoxon(v, alternative='two-sided')
                    ax.text(0.02, 0.98, f'p={p0:.4g}', transform=ax.transAxes, va='top', fontsize=8)
                except Exception:
                    pass
            plt.tight_layout()
            fig4.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig4_acc_hpc_masked_subject_bars.pdf'), dpi=300, bbox_inches='tight')
            fig4.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig4_acc_hpc_masked_subject_bars.png'), dpi=300, bbox_inches='tight')
            plt.show()

            labels8 = ['rho_full', 'rho_item', 'ri_cat', 'ri_item']
            cols8 = [m[0] for m in metrics]
            means, lows, highs = [], [], []
            rng = np.random.default_rng(42)
            for col in cols8:
                v = dfm[col].dropna().to_numpy(dtype=float)
                if len(v) == 0:
                    means.append(np.nan)
                    lows.append(np.nan)
                    highs.append(np.nan)
                    continue
                means.append(float(np.mean(v)))
                boots = [rng.choice(v, size=len(v), replace=True).mean() for _ in range(1000)]
                boots = np.asarray(boots)
                lows.append(float(np.percentile(boots, 2.5)))
                highs.append(float(np.percentile(boots, 97.5)))
            fig8, ax8 = plt.subplots(figsize=(8, 4))
            x8 = np.arange(len(labels8))
            marr, larr, harr = np.array(means), np.array(lows), np.array(highs)
            err = [marr - larr, harr - marr]
            ax8.bar(x8, marr, yerr=err, capsize=4, color='steelblue', edgecolor='k')
            ax8.set_xticks(x8)
            ax8.set_xticklabels(labels8)
            ax8.axhline(0, color='k', linewidth=0.8)
            plt.tight_layout()
            fig8.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig8_acc_hpc_headline_bars.pdf'), dpi=300, bbox_inches='tight')
            fig8.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig8_acc_hpc_headline_bars.png'), dpi=300, bbox_inches='tight')
            plt.show()

        # ----- Figure 5: marginals
        rf = np.nanmean(stacks['rho_full_diff'], axis=0)
        inside = sig_mask_z & np.isfinite(rf)
        if np.any(inside):
            peak_idx = np.unravel_index(np.nanargmax(np.where(inside, rf, np.nan)), rf.shape)
        else:
            peak_idx = np.unravel_index(np.nanargmax(rf), rf.shape)
        t_fb_star, t_cue_star = int(peak_idx[0]), int(peak_idx[1])

        fig5, axes5 = plt.subplots(2, 1, figsize=(8, 8))
        for name in diff_names:
            D = stacks[name]
            if D is None:
                continue
            m = np.nanmean(D, axis=0)
            axes5[0].plot(fb_t, m[:, t_cue_star], label=name, alpha=0.85)
        axes5[0].axvline(fb_t[t_fb_star], color='k', linestyle=':', alpha=0.5)
        axes5[0].set_xlabel(f'Feedback ({ENC_ROI})')
        axes5[0].legend(fontsize=7)
        for name in diff_names:
            D = stacks[name]
            if D is None:
                continue
            m = np.nanmean(D, axis=0)
            axes5[1].plot(cue_t, m[t_fb_star, :], label=name, alpha=0.85)
        axes5[1].axvline(cue_t[t_cue_star], color='k', linestyle=':', alpha=0.5)
        axes5[1].set_xlabel(f'Cue ({CUE_ROI})')
        axes5[1].legend(fontsize=7)
        plt.tight_layout()
        fig5.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig5_acc_hpc_marginals.pdf'), dpi=300, bbox_inches='tight')
        fig5.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig5_acc_hpc_marginals.png'), dpi=300, bbox_inches='tight')
        plt.show()

        # ----- Figure 6: example subject RDMs
        try:
            ex_rank = rank_example_subjects_for_pair(
                trial_save_dir=RSA_TRIAL_SAVE_DIR,
                group_save_dir=RSA_GROUP_SAVE_DIR,
                roi_pair=(ENC_ROI, CUE_ROI),
            )
            if len(ex_rank) > 0:
                ex_subj = str(ex_rank.iloc[0]['subject'])
                s_obj = Subject(ex_subj, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)
                ex_data = load_subject_crossregion_hfa(
                    subject_obj=s_obj,
                    encoding_roi=ENC_ROI,
                    cue_roi=CUE_ROI,
                    event_encoding=EVENT_ENCODING,
                    event_cue=EVENT_CUE,
                    freq_range=FREQ_RANGE,
                    downsample_factor=DOWNSAMPLE_FACTOR,
                    win_ms=WIN_MS,
                    min_trials=MIN_TRIALS,
                )
                rpe_ex = np.asarray(ex_data['rpe_categories'], dtype=object)
                hit_m = ex_data['memory_outcomes'] == 'remembered'
                labs = np.array([_canonicalize_rpe_cat_label(x) or 'na' for x in rpe_ex[hit_m]])
                order = np.argsort(pd.Categorical(labs, categories=['negative', 'neutral', 'positive', 'na']))
                enc_h = ex_data['encoding_hfa'][hit_m][order][:, :, t_fb_star]
                cue_h = ex_data['cue_hfa'][hit_m][order][:, :, t_cue_star]
                v_acc = _compute_pairwise_dissimilarity(enc_h, method=RDM_METHOD)
                v_hpc = _compute_pairwise_dissimilarity(cue_h, method=RDM_METHOD)
                rdm_acc = squareform(v_acc) if v_acc.size else np.zeros((0, 0))
                rdm_hpc = squareform(v_hpc) if v_hpc.size else np.zeros((0, 0))
                S_ex = _padded_row_zscore_cosine_similarity(enc_h, cue_h)
                M_ex = build_rpe_category_model_rdm(rpe_ex[hit_m][order])
                fig6, ax6 = plt.subplots(1, 4, figsize=(16, 3.5))
                vmax6 = max(np.nanmax(rdm_acc) if rdm_acc.size else 0, 1e-6)
                ax6[0].imshow(rdm_acc, cmap='viridis', vmin=0, vmax=vmax6)
                ax6[0].set_title('ACC RDM')
                ax6[1].imshow(rdm_hpc, cmap='viridis', vmin=0, vmax=vmax6)
                ax6[1].set_title('HPC RDM')
                ax6[2].imshow(S_ex, cmap='RdBu_r', vmin=SHARED_CBAR_MIN, vmax=SHARED_CBAR_MAX)
                ax6[2].set_title('S')
                vmn = np.nanmin(M_ex)
                vmx = np.nanmax(M_ex)
                ax6[3].imshow(M_ex, cmap='Greys', vmin=vmn, vmax=vmx)
                ax6[3].set_title('M_rpe')
                for ax in ax6:
                    ax.set_xticks([])
                    ax.set_yticks([])
                plt.suptitle(f'{ex_subj} | sorted by rpe_cat', fontsize=10)
                plt.tight_layout()
                fig6.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig6_acc_hpc_example_rdms_category_sorted.pdf'), dpi=300, bbox_inches='tight')
                fig6.savefig(os.path.join(ACC_HPC_PLOT_DIR, 'fig6_acc_hpc_example_rdms_category_sorted.png'), dpi=300, bbox_inches='tight')
                plt.show()
        except Exception as e6:
            print('Figure 6 skipped:', e6)

        # ----- Figure 7: small multiples
        n_s = stacks['rho_item_diff'].shape[0]
        ncol = max(1, int(np.ceil(np.sqrt(n_s))))
        nrow = int(np.ceil(n_s / ncol))

        def _plot_small_mult(stack, name, fname_suffix):
            if n_s < 1:
                return
            vmax = _vmax99_list([np.nanmean(stack, axis=0)])
            fig, axes = plt.subplots(nrow, ncol, figsize=(2 * ncol, 2 * nrow))
            axes = np.atleast_2d(axes)
            for i in range(n_s):
                r, c = i // ncol, i % ncol
                ax = axes[r, c]
                g = stack[i]
                ax.imshow(
                    g, origin='lower', aspect='auto', extent=extent,
                    cmap='RdBu_r', vmin=SHARED_CBAR_MIN, vmax=SHARED_CBAR_MAX,
                )
                if sig_mask_z.shape == g.shape:
                    ax.contour(cue_t, fb_t, sig_mask_z.astype(int), levels=[0.5], colors='k', linewidths=0.6)
                ax.set_title(loaded_p[i], fontsize=7)
                ax.set_xticks([])
                ax.set_yticks([])
            for j in range(n_s, nrow * ncol):
                rj, cj = j // ncol, j % ncol
                axes[rj, cj].axis('off')
            plt.suptitle(name, fontsize=11)
            plt.tight_layout()
            fig.savefig(os.path.join(ACC_HPC_PLOT_DIR, fname_suffix), dpi=300, bbox_inches='tight')
            fig.savefig(os.path.join(ACC_HPC_PLOT_DIR, fname_suffix.replace('.pdf', '.png')), dpi=300, bbox_inches='tight')
            plt.show()

        _plot_small_mult(stacks['rho_item_diff'], 'rho_item_diff', 'fig7_acc_hpc_smallmult_rho_item_diff.pdf')
        _plot_small_mult(stacks['ri_cat_diff'], 'ri_cat_diff', 'fig7_acc_hpc_smallmult_ri_cat_diff.pdf')

        # ----- Combined PDF from PNGs
        from matplotlib.backends.backend_pdf import PdfPages as _Pdf
        from matplotlib.image import imread
        out_comb = os.path.join(ACC_HPC_PLOT_DIR, 'acc_hpc_reactivation_content_report.pdf')
        pngs = sorted(glob.glob(os.path.join(ACC_HPC_PLOT_DIR, 'fig*.png')))
        if pngs:
            with _Pdf(out_comb) as pdf:
                for png in pngs:
                    im = imread(png)
                    fc, axc = plt.subplots(figsize=(12, 10))
                    axc.imshow(im)
                    axc.axis('off')
                    pdf.savefig(fc, bbox_inches='tight')
                    plt.close(fc)
            print('Saved', out_comb)
        else:
            # at least fig1 has png
            print('No fig*.png for combined PDF; save fig1 as png first')


In [ ]:
ACC_HPC_PLOT_DIR

## Per-subject RSA z-diff (masked by group cluster)

For each participant, **mean z(diff RSA)** over pixels inside the full-cohort **z-cluster significance mask** (`sig_mask_z` from the group z-diff test). Bar plot shows subject-level consistency of the effect in the same region used for inference.


In [ ]:
# =============================
# Per-subject RSA z-diff within group z-cluster mask (sig_mask_z)
# =============================
SUBJ_MASKED_Z_OUT = os.path.join(RSA_CONTROLS_DIR, 'per_subject_masked_z')
os.makedirs(SUBJ_MASKED_Z_OUT, exist_ok=True)

if 'z_stack' not in globals() or z_stack is None or (hasattr(z_stack, 'shape') and z_stack.shape[0] == 0):
    meta_path = os.path.join(Z_CLUSTER_OUT, 'ACC_HPC_z_cluster_meta.npz')
    if os.path.exists(meta_path):
        mz = np.load(meta_path, allow_pickle=True)
        z_stack = np.asarray(mz['z_stack'], dtype=float)
        z_subjects = [str(x) for x in np.asarray(mz['subjects_included'])]
        if 'sig_mask_z' not in globals() or sig_mask_z is None:
            sig_mask_z = np.asarray(mz['sig_mask_z'], dtype=bool)
    else:
        z_stack = np.empty((0, 0, 0))
        z_subjects = []

if z_stack.shape[0] >= 1 and sig_mask_z is not None and np.any(sig_mask_z):
    Z = np.asarray(z_stack, dtype=np.float64)
    m = np.asarray(sig_mask_z, dtype=bool)
    if Z.shape[1:] != m.shape:
        raise ValueError('sig_mask_z shape must match z_stack spatial dimensions')

    per_subj = np.array([float(np.nanmean(Z[i][m])) for i in range(Z.shape[0])])
    group_mean = float(np.nanmean(per_subj))

    df = pd.DataFrame({'participant': z_subjects, 'masked_mean_z_diff': per_subj})
    csv_path = os.path.join(SUBJ_MASKED_Z_OUT, 'per_subject_masked_z_diff_ACC_HPC.csv')
    df.to_csv(csv_path, index=False)

    fig, ax = plt.subplots(figsize=(max(8.0, 0.42 * len(z_subjects)), 4.5))
    x = np.arange(len(z_subjects))
    ax.bar(x, per_subj, color='steelblue', alpha=0.85, edgecolor='k', linewidth=0.5)
    ax.axhline(0.0, color='k', linestyle='-', linewidth=0.8)
    ax.axhline(
        group_mean,
        color='coral',
        linestyle='--',
        linewidth=1.5,
        label=f'group mean = {group_mean:.3f}',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(z_subjects, rotation=45, ha='right')
    ax.set_ylabel('mean z(diff RSA) inside sig_mask_z')
    ax.set_xlabel('Participant')
    ax.set_title('Per-subject RSA z-diff (group z-cluster mask)\nACC→HPC')
    ax.legend(loc='upper right')
    plt.tight_layout()
    pdf_path = os.path.join(SUBJ_MASKED_Z_OUT, 'per_subject_masked_z_diff_ACC_HPC.pdf')
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(
        f'Saved {csv_path} and {pdf_path} | n={len(z_subjects)} | '
        f'group mean masked z = {group_mean:.4f}'
    )
else:
    print('Skipped: need z_stack with >=1 subject and non-empty sig_mask_z.')


In [ ]:
z_stack

In [ ]:
sig_mask_z

In [ ]:
# =============================
# Group-level cluster test + plot
# =============================
pair_paths = build_crossregion_rsa_paths(
    trial_save_dir=RSA_TRIAL_SAVE_DIR,
    group_save_dir=RSA_GROUP_SAVE_DIR,
    encoding_roi=ROI_PAIRS[2][0],
    cue_roi=ROI_PAIRS[2][1],
    subject=None,
)

group_npz_path = pair_paths['group_npz']
if os.path.exists(group_npz_path):
    g = np.load(group_npz_path, allow_pickle=True)
    group_mean_diff = np.asarray(g['group_mean_diff'])
    feedback_times = np.asarray(g['feedback_times'])
    cue_times = np.asarray(g['cue_times'])
    # cluster_sig_mask = np.asarray(g['cluster_sig_mask'])
    # cluster_pvals = np.asarray(g['cluster_pvals'])
    subjects_included = np.asarray(g['subjects_included'])

    cluster_pvals=np.asarray(cluster_res['cluster_pvals'])
    cluster_sig_mask=np.asarray(cluster_res['sig_mask']) if cluster_res['sig_mask'] is not None else np.array([])

    if group_mean_diff.size == 0:
        print('Group NPZ has empty group_mean_diff.')
    else:
        vmax = np.nanmax(np.abs(group_mean_diff))
        if (not np.isfinite(vmax)) or vmax == 0:
            vmax = 1.0

        plt.figure(figsize=(6, 5))
        plt.imshow(
            group_mean_diff,
            origin='lower',
            aspect='auto',
            extent=[float(cue_times.min()), float(cue_times.max()), float(feedback_times.min()), float(feedback_times.max())],
            cmap='RdBu_r',
            vmin=-vmax,
            vmax=vmax,
        )
        if cluster_sig_mask.size > 0 and np.any(cluster_sig_mask):
            plt.contour(
                cue_times,
                feedback_times,
                cluster_sig_mask.astype(int),
                levels=[0.5],
                colors='k',
                linewidths=1.5,
            )
        plt.title(f'Group RSA mean diff\n{ROI_PAIRS[2][0]} -> {ROI_PAIRS[2][1]}')
        plt.xlabel('Cue Time (s)')
        plt.ylabel('Feedback Time (s)')
        # plt.colorbar(shrink=0.8)
        plt.tight_layout()
        # Save plot
        save_path = f'{RSA_GROUP_SAVE_DIR}/cross_temporal_rsa_cluster_{ROI_PAIRS[2][0]}_{ROI_PAIRS[2][1]}.pdf'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        # plt.show()

        sig_pvals = cluster_pvals[cluster_pvals <= 0.05] if cluster_pvals.size > 0 else np.array([])
        print('Included subjects:', len(subjects_included))
        print('n_clusters:', len(cluster_pvals), '| n_sig_clusters:', len(sig_pvals))
        if len(sig_pvals) > 0:
            print('sig p-values:', np.sort(sig_pvals))
        else:
            print('No significant clusters at p<=0.05')
else:
    print('Group NPZ not found:', group_npz_path)


## Output Index

In [ ]:
# =============================
# Output index
# =============================
print('Trial-level RSA root:')
print(' ', RSA_TRIAL_SAVE_DIR)

print('')
print('Group-level RSA root:')
print(' ', RSA_GROUP_SAVE_DIR)

RSA_CONTROLS_DIR = os.path.join(mpaths.results_root(), 'rsa_controls')
print('')
print('RSA controls (z-cluster, cross-temporal RI, per-subject masked z):')
print(' ', RSA_CONTROLS_DIR)

group_summary_path = os.path.join(RSA_GROUP_SAVE_DIR, 'group_summary_crossregion_rsa.csv')
print('')
print('Group summary CSV:')
print(' ', group_summary_path, '| exists:', os.path.exists(group_summary_path))

for enc_roi, cue_roi in ROI_PAIRS:
    paths = build_crossregion_rsa_paths(
        trial_save_dir=RSA_TRIAL_SAVE_DIR,
        group_save_dir=RSA_GROUP_SAVE_DIR,
        encoding_roi=enc_roi,
        cue_roi=cue_roi,
        subject=None,
    )
    n_subject_files = len(glob.glob(os.path.join(paths['pair_dir'], '*_rsa.npz')))
    print(f"{enc_roi}->{cue_roi}: subject_npz={n_subject_files}, group_npz_exists={os.path.exists(paths['group_npz'])}")
